# Merge Data Sets and Clean

In [521]:
import pandas as pd
import numpy as np
import glob
import os
import re, math, unicodedata
from pathlib import Path

In [ ]:
# Define file paths
PATH_SELLER_A = 'data/seller-a/'
PATH_SELLER_B = 'data/seller-b/nish_catalog_run.csv'
PATH_SELLER_C = 'data/seller-c/jngems_custom_ignore.csv'
PATH_SELLER_D = 'data/seller-d/ratnapura.csv'
PATH_MASTER_DATASET = 'data/master_gem_schema.csv'
PATH_MASTER_CLEAN_DATASET = 'data/master_gem_schema_clean.csv'

# Define currency conversion rate
USD_TO_LKR = 330.0

# Define regex pattern for identifying pairs or sets
PAIR_PAT = re.compile(r"\b(pair|set|lot|parcel|pcs|pieces|two\s*stones?|matching|couple)\b", re.I)

# Initialize list to hold dataframes
frames = []

PATH_SELLER_A = 'data/seller-a/master-a.csv'


## Merge All Seller A Data set into one master

In [ ]:
all_files = glob.glob(os.path.join(PATH_SELLER_A, '*.csv'))

# dataframes = []
# for file in all_files:
#     df = pd.read_csv(file)
#     dataframes.append(df)

# merged_df = pd.concat(dataframes, ignore_index=True)
# # Save the merged DataFrame to a new CSV file
# merged_df.to_csv(PATH_SELLER_A + 'master-a.csv', index=False, mode='w')
# print("Merged Seller A DataFrame. Shape:", merged_df.shape)

# PATH_SELLER_A = 'data/seller-a/master-a.csv'
# df_a = pd.read_csv(PATH_SELLER_A)
# df_a.head()


# # Exclude the master file if it already exists (to avoid self-merging)
# master_file = os.path.join(PATH_SELLER_A, 'master-a.csv')
# all_files = [f for f in all_files if f != master_file]

# # Merge all CSVs
# dataframes = [pd.read_csv(file) for file in all_files]
# merged_df = pd.concat(dataframes, ignore_index=True)

# # If master file exists, delete it first (optional safety step)
# if os.path.exists(master_file):
#     os.remove(master_file)

# # Save new master file (always overwrite)
# merged_df.to_csv(master_file, index=False)
# print("Merged Seller A DataFrame. Shape:", merged_df.shape)

# # Reload for check
# df_a = pd.read_csv(master_file)
# print(df_a.head())

# PATH_SELLER_A = 'data/seller-a/master-a.csv'


### Helper Methods
Helper functions for data normalization and cleaning

In [524]:
def _norm_text(x):
    if pd.isna(x): return x
    s = str(x)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0"," ").replace("\u200B","")
    s = re.sub(r"\s+"," ", s).strip()
    return s

def _to_float(x):
    if pd.isna(x) or x == "": return np.nan
    if isinstance(x,(int,float)): return float(x)
    s = re.sub(r"[^\d.\-eE]", "", str(x).strip())
    try: return float(s)
    except: return np.nan

def parse_dimensions_mm(s):
    # '10.1 x 8.4 x 6.8 mm' -> (10.1, 8.4, 6.8)
    if pd.isna(s): return np.nan, np.nan, np.nan
    s = _norm_text(s).lower().replace("mm","")
    parts = re.split(r"[x×]", s)
    nums = []
    for p in parts:
        v = _to_float(p)
        if not (pd.isna(v) or np.isnan(v)): nums.append(v)
    while len(nums) < 3: nums.append(np.nan)
    return nums[0], nums[1], nums[2]

def fmt_dims(l, w, h):
    vals = [l, w, h]
    ok = [v for v in vals if not (pd.isna(v) or np.isnan(v))]
    if not ok: return ""
    txt = " x ".join([f"{v:.2f}".rstrip("0").rstrip(".") for v in ok]) + " mm"
    return txt

def series_or(df, col, fill=np.nan):
    return df[col] if col in df.columns else pd.Series([fill]*len(df), index=df.index)

def cert_yesno_from_row(row, scan_cols):
    # “Yes” if any column hints a cert; else “No”
    KEYS = ("cert","certificate","gia","igi","gic","ngja","ngtc")
    for c in scan_cols:
        if c in row.index and pd.notna(row[c]):
            s = str(row[c]).lower()
            if any(k in s for k in KEYS): 
                return "Yes"
    # explicit numeric flag also counts
    for c in ("certificate_flag",):
        if c in row.index and pd.notna(row[c]):
            try:
                if int(row[c]) == 1: 
                    return "Yes"
            except:
                pass
    return "No"

def date_from_any(df, candidates=("scraped_at","date","Date","created_at","updated_at")):
    for c in candidates:
        if c in df.columns:
            dt = pd.to_datetime(df[c], errors="coerce")
            if dt.notna().any():
                return dt.dt.date.astype("string")
    return pd.Series([""]*len(df), index=df.index, dtype="string")

def split_gemtype_variety(text):
    """
    Returns (Gem Type, Variety).
    If text contains 'sapphire' -> Gem Type='Sapphire', Variety=Title(text) (ensure ends with 'Sapphire').
    Else Gem Type and Variety both = Title(text) (e.g., 'Ruby', 'Emerald').
    """
    if pd.isna(text) or str(text).strip()=="":
        return "", ""
    t = _norm_text(str(text)).replace("-", " ").strip().lower()
    vt = t.title()
    if "sapphire" in t:
        # ensure 'Sapphire' suffix for colours like 'Padparadscha'
        if "sapphire" not in vt.lower():
            vt = vt + " Sapphire"
        return "Sapphire", vt
    # common single-type gems
    for base in ["ruby","emerald","spinel","alexandrite","tourmaline","topaz","aquamarine","garnet"]:
        if base in t:
            return base.title(), vt
    # fallback
    return vt, vt

def build_sellers_schema(df, source_hint="", colmap=None, scan_cert_cols=None, dim_cols=None):
    """
    Returns a DF with exactly the sellers columns + Date:
    Gem Type, Variety, Carat Weight (ct), Cut/Shape, Colour, Clarity, Treatments,
    Asking Price (LKR), Final Selling Price (if available), Dimensions (LxWxH, mm),
    Certification Status, Date
    """
    colmap = colmap or {}
    scan_cert_cols = scan_cert_cols or []
    dim_cols = dim_cols or {}

    # ---- Source-specific raw columns ----
    name = series_or(df, colmap.get("name","name"), "")
    category = series_or(df, colmap.get("category","category"), "")
    gem_type_raw = series_or(df, colmap.get("gem_type","gem_type"), "")
    # prefer explicit gem_type/category, else fall back to name
    base_txt = gem_type_raw.where(gem_type_raw.astype(str).str.len()>0, category)
    base_txt = base_txt.where(base_txt.astype(str).str.len()>0, name)

    GT, V = [], []
    for s in base_txt.fillna(""):
        g, v = split_gemtype_variety(s)
        GT.append(g); V.append(v)

    # weights
    wt = series_or(df, colmap.get("weight","weight_carat"), np.nan).map(_to_float)

    # shape/colour/clarity/treatments
    shape = series_or(df, colmap.get("shape","shape"), "")
    colour = series_or(df, colmap.get("colour","colour"), "")
    clarity = series_or(df, colmap.get("clarity","clarity"), "")
    treat = series_or(df, colmap.get("treatment","treatment"), "")

    # price LKR
    price_col = colmap.get("price_lkr")
    price = series_or(df, price_col, np.nan).map(_to_float)

    # final selling price (usually absent)
    final_price = series_or(df, colmap.get("final_price",""), "").map(_to_float)

    # dimensions
    L = series_or(df, dim_cols.get("L","length_mm"), np.nan).map(_to_float)
    W = series_or(df, dim_cols.get("W","width_mm"), np.nan).map(_to_float)
    H = series_or(df, dim_cols.get("H","depth_mm"), np.nan).map(_to_float)
    dim_str = [fmt_dims(l,w,h) for l,w,h in zip(L,W,H)]

    # if no L/W, try parsing size text
    if (pd.isna(L).all() and pd.isna(W).all()):
        size_txt = series_or(df, dim_cols.get("size","size_mm"), "")
        dims = size_txt.apply(parse_dimensions_mm)
        dim_str = [fmt_dims(a,b,c) for (a,b,c) in dims]

    # certificate status
    cert = [cert_yesno_from_row(df.loc[i], scan_cert_cols) for i in df.index]

    # date
    date_ser = date_from_any(df, candidates=colmap.get("date_candidates", ("scraped_at","date","Date","created_at","updated_at")))

    # build sellers schema
    out = pd.DataFrame({
        "Gem Type": GT,
        "Variety": V,
        "Carat Weight (ct)": wt.round(2),
        "Cut/Shape": shape.map(lambda s: _norm_text(s).title() if isinstance(s,str) else ""),
        "Colour": colour.map(lambda s: _norm_text(s).title() if isinstance(s,str) else ""),
        "Clarity": clarity.map(lambda s: _norm_text(s).upper() if isinstance(s,str) else ""),
        "Treatments": treat.map(lambda s: _norm_text(s).title() if isinstance(s,str) else ""),
        "Asking Price (LKR)": price,
        "Final Selling Price (if available)": final_price,   # keep blank if unknown
        "Dimensions (LxWxH, mm)": dim_str,
        "Certification Status": cert,  # Yes/No
        "Date": date_ser.fillna("").astype(str),
    })

    # remove pair/set rows using name/category if available
    name_for_filter = name.where(name.astype(str).str.len()>0, category)
    mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)
    out = out[~mask_pair].copy()

    # tidy blanks
    out["Gem Type"] = out["Gem Type"].fillna("")
    out["Variety"] = out["Variety"].fillna("")
    out["Certification Status"] = out["Certification Status"].replace({"": "No"})

    return out

# Read & normalize each source into the SAME schema

### Final Expected Gem Dataset Column Structure

| Gem Type | Variety | Carat Weight (ct) | Cut/Shape | Colour | Clarity | Treatments | Asking Price (LKR) | Final Selling Price (LKR) | Dimensions (LxWxH, mm) | Certification Status | Date |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| *e.g., Corundum* | *e.g., Ruby* | *e.g., 1.50* | *e.g., Cushion* | *e.g., Vivid Red* | *e.g., SI* | *e.g., Heat* | *e.g., 250,000* | *e.g., 230,000* | *e.g., 7.2x5.8x3.9* | *e.g., GIC, EGL, None* | *e.g., 2023-11-15* |


## Seller - C

In [525]:
# JN-specific heated normalizer
def jn_map_heated(val):
        # Your rule: blank/NaN => Unheated
        if pd.isna(val) or str(val).strip() == "":
            return "Unheated"
        v = str(val).strip().lower()
        if v in ("yes","y","1","true"):
            return "Heated"
        if v in ("no","n","0","false"):
            return "Unheated"
        # if they sometimes write phrases like "no heat" / "unheated" etc.
        if "no heat" in v or "unheated" in v or v == "none":
            return "Unheated"
        if "heat" in v:
            return "Heated"
        # fallback to your general normalizer (in case of odd strings)
        return v

# JN-specific variety fixer for sapphires
def _primary_hue_from_colour(col):
        if pd.isna(col): return ""
        s = str(col).lower()
        # padparadscha first (pink+orange family)
        if "padpar" in s or (("pink" in s) and ("orange" in s or "peach" in s)):
            return "Padparadscha"
        if "teal" in s:        return "Teal"
        if "blue" in s:        return "Blue"
        if "pink" in s:        return "Pink"
        if "yellow" in s:      return "Yellow"
        if "green" in s:       return "Green"
        if "violet" in s or "purple" in s: return "Purple"
        if "orange" in s:      return "Orange"
        if "peach" in s:       return "Peach"
        if "white" in s or "colorless" in s or "colourless" in s: return "White"
        return ""

# JN-specific variety fixer for sapphires
def jn_variety_from_gem_and_colour(gem_type, colour, current_variety):
        gt = "" if pd.isna(gem_type) else str(gem_type).lower()
        var = "" if pd.isna(current_variety) else str(current_variety).strip()
        # only intervene for sapphires, and when variety is blank or 'Ceylon'
        if "sapphire" not in gt:
            return current_variety  # leave other gem types as-is
        if var == "" or re.search(r"\bceylon\b", var, flags=re.I):
            hue = _primary_hue_from_colour(colour)
            return (hue + " Sapphire") if hue else "Sapphire"
        return current_variety

if Path(PATH_SELLER_C).exists():
    jn = pd.read_csv(PATH_SELLER_C)

    jn["heated_mapped"] = jn["heated"].apply(jn_map_heated)
    jn_sellers = build_sellers_schema(
        jn,
        colmap={
            "name": "title",
            "category": "category_slug",
            "gem_type": "category_slug",
            "weight": "weight_carat",
            "shape": "shape_cut",
            "colour": "colour",
            "clarity": "clarity",
            "treatment": "heated_mapped",
            "price_lkr": "price_current_value",
            "date_candidates": ("scraped_at",),
        },
        scan_cert_cols=["certificate","product_description"]
    )

    mask_jn = jn_sellers["Gem Type"].str.contains("Sapphire", case=False, na=False)
    jn_sellers.loc[mask_jn, "Variety"] = jn_sellers.loc[mask_jn].apply(
        lambda r: jn_variety_from_gem_and_colour(r["Gem Type"], r["Colour"], r["Variety"]), axis=1
    )

    frames.append(jn_sellers)
    

/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


## Seller - B

In [526]:
if Path(PATH_SELLER_B).exists():
    ng = pd.read_csv(PATH_SELLER_B)
    # convert price to LKR first
    if "price_value" in ng.columns:
        ng["price_lkr"] = ng["price_value"].map(_to_float) * USD_TO_LKR
    frames.append(build_sellers_schema(
        ng,
        colmap={
            "name": "name",
            "category": "category",
            "gem_type": "category",
            "weight": "weight_cts",
            "shape": "shape",
            "colour": "colour",
            "clarity": "clarity",
            "treatment": "treatment",
            "price_lkr": "price_lkr",
            "date_candidates": ("collected_at",),
        },
        scan_cert_cols=["certificate","description"],
        dim_cols={"size": "dimensions"}  # Nish has free-text dimensions
    ))

/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


## Seller - A

In [527]:
if Path(PATH_SELLER_A).exists():
    wz = pd.read_csv(PATH_SELLER_A)
    seller_a = build_sellers_schema(
        wz,
        colmap={
            "name": "name",
            "category": "gem_type",
            "gem_type": "gem_type",
            "weight": "weight_carat",
            "shape": "shape_cut",
            "colour": "colour",
            "clarity": "clarity",
            "treatment": "treatment",
            "price_lkr": "price_value",
            "date_candidates": ("collected_at",),
        },
        scan_cert_cols=["price_text","raw_description_text"],
        dim_cols={"size": "size_mm"}
    )

    frames.append(seller_a)


/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


In [528]:
frames

[      Gem Type                Variety  Carat Weight (ct)  \
 0     Sapphire          Blue Sapphire               5.88   
 1     Sapphire          Blue Sapphire               0.86   
 2     Sapphire  Padparadscha Sapphire               0.77   
 3     Sapphire               Sapphire               1.24   
 4     Sapphire          Blue Sapphire               1.77   
 ..         ...                    ...                ...   
 257   Ametrine               Ametrine                NaN   
 258   Ametrine               Ametrine                NaN   
 259   Sapphire     Bi Color Sapphires               2.15   
 260   Sapphire     Bi Color Sapphires               4.54   
 263  Moonstone              Moonstone               1.11   
 
                     Cut/Shape  \
 0             Cushion Mix Cut   
 1                  Oval ,Step   
 2                   Oval Step   
 3                   Oval Step   
 4                   Oval Step   
 ..                        ...   
 257              Octagon St

## Seller - D

In [529]:
if Path(PATH_SELLER_D).exists():
    sd = pd.read_csv(PATH_SELLER_D)
    # Try to align columns if already in Sellers format
    # If your sheet already has the exact names, just rename/keep
    possible = sd.copy()
    # unify headings if slightly different
    rename_map = {
        "GemType": "Gem Type",
        "Gem type": "Gem Type",
        "Variety ": "Variety",
        "Carat Weight": "Carat Weight (ct)",
        "Cut/Shape ": "Cut/Shape",
        "Color": "Colour",
        "Treatment": "Treatments",
        "Asking Price (LKR) ": "Asking Price (LKR)",
        "Final Selling Price": "Final Selling Price (if available)",
        "Certification": "Certification Status",
        "Dimensions": "Dimensions (LxWxH, mm)",
        "Date ": "Date",
    }
    possible.rename(columns=rename_map, inplace=True)
    needed = ["Gem Type","Variety","Carat Weight (ct)","Cut/Shape","Colour","Clarity","Treatments",
              "Asking Price (LKR)","Final Selling Price (if available)","Dimensions (LxWxH, mm)","Certification Status","Date"]
    if set(needed).issubset(set(possible.columns)):
        # just take & coerce types
        chunk = possible[needed].copy()
        # normalize types a bit
        chunk["Asking Price (LKR)"] = chunk["Asking Price (LKR)"].map(_to_float)
        chunk["Final Selling Price (if available)"] = chunk["Final Selling Price (if available)"].map(_to_float)
        frames.append(chunk)
    else:
        # Fallback: build from raw with flexible scan of cert & dimension columns
        frames.append(build_sellers_schema(
            sd,
            colmap={
                "name": "name",
                "category": "Gem Type",
                "gem_type": "Gem Type",
                "weight": "Carat Weight (ct)",
                "shape": "Cut/Shape",
                "colour": "Colour",
                "clarity": "Clarity",
                "treatment": "Treatments",
                "price_lkr": "Asking Price (LKR)",
                "final_price": "Final Selling Price (if available)",
                "date_candidates": ("Date","scraped_at"),
            },
            scan_cert_cols=list(sd.columns),
            dim_cols={"size": "Dimensions (LxWxH, mm)"}
        ))

/var/folders/x0/kj7rxcb54sb06z9f6ylxnt000000gn/T/ipykernel_9841/485891275.py:162: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_pair = name_for_filter.astype(str).str.contains(PAIR_PAT, na=False)


In [530]:
if not frames:
    raise RuntimeError("No input files found. Check the PATH_* variables at the top.")
# Combine all frames
master = pd.concat(frames, ignore_index=True)

# basic final tidy
master["Certification Status"] = master["Certification Status"].replace({"": "No"}).fillna("No")
master["Asking Price (LKR)"] = master["Asking Price (LKR)"].map(_to_float)
master["Final Selling Price (if available)"] = master["Final Selling Price (if available)"].map(_to_float)
master["Date"] = master["Date"].fillna("").astype(str)

# keep EXACT column order
cols = ["Gem Type","Variety","Carat Weight (ct)","Cut/Shape","Colour","Clarity","Treatments",
        "Asking Price (LKR)","Final Selling Price (if available)","Dimensions (LxWxH, mm)","Certification Status","Date"]
master = master[cols].reset_index(drop=True)

# remove totally empty rows (no Gem Type & no Dimensions)
mask_empty = (master["Gem Type"].astype(str).str.strip()=="") & (master["Dimensions (LxWxH, mm)"].astype(str).str.strip()=="")
master = master[~mask_empty].copy()

OUT = Path(PATH_MASTER_DATASET)
master.to_csv(OUT, index=False)

print(f"Saved -> {OUT.resolve()}")
print("Rows:", len(master))
print(master.head(10))

Saved -> /Users/helithasri/Personal/Gem Price Prediction System Ratnapura Srilanka/notebooks/data/master_gem_schema.csv
Rows: 599
   Gem Type                Variety  Carat Weight (ct)            Cut/Shape  \
0  Sapphire          Blue Sapphire               5.88      Cushion Mix Cut   
1  Sapphire          Blue Sapphire               0.86           Oval ,Step   
2  Sapphire  Padparadscha Sapphire               0.77            Oval Step   
3  Sapphire               Sapphire               1.24            Oval Step   
4  Sapphire          Blue Sapphire               1.77            Oval Step   
5  Sapphire               Sapphire               1.18           Pear ,Step   
6  Sapphire         White Sapphire               1.23  Asscher Diamond Cut   
7  Sapphire          Blue Sapphire               1.55            Oval Step   
8  Sapphire          Blue Sapphire               1.88            Oval Step   
9  Sapphire          Blue Sapphire               2.24            Oval Step   

           

In [531]:
master.isnull().sum()

Gem Type                                0
Variety                                 0
Carat Weight (ct)                      25
Cut/Shape                               0
Colour                                  0
Clarity                                 0
Treatments                              0
Asking Price (LKR)                      0
Final Selling Price (if available)    583
Dimensions (LxWxH, mm)                  0
Certification Status                    0
Date                                    0
dtype: int64

# Final Cleaning

In [ ]:
df = pd.read_csv(PATH_MASTER_DATASET)

# Normalize text columns
for c in ["Gem Type","Variety","Cut/Shape","Colour","Clarity","Treatments",
          "Certification Status","Dimensions (LxWxH, mm)","Date"]:
    if c in df.columns:
        df[c] = df[c].map(_norm_text)   # reuse your helper

# Gem Type / Variety cleanup
# Normalize Variety by removing words like 'Natural', 'AAA', etc.
def tidy_variety(v):
    v = _norm_text(v).title()
    v = re.sub(r"\b(Natural|100%|Genuine|Premium|Top|AAA\+?|A\+)\b", "", v, flags=re.I)
    v = re.sub(r"\s+", " ", v).strip()
    return v

# Special handling for Sapphire varieties
# def split_sapphire(gt, v):
#     gt = _norm_text(gt).title()
#     v = tidy_variety(v)
#     combined = (gt + " " + v).lower()
#     if "sapphire" in combined:
#         m = re.search(r"(padparadscha|teal|pink|yellow|blue|white|green|purple|peach)\s*sapphire", v, flags=re.I)
#         if m:
#             return "Sapphire", m.group(1).title()
#         v = re.sub(r"\s*Sapphire\b", "", v, flags=re.I).strip()
#         return "Sapphire", v if v else "Sapphire"
#     return (gt if gt else v) or "", v

def split_sapphire(gt, v):
    """
    Keep Variety like 'Blue Sapphire' if already present.
    Otherwise, if Variety contains '<hue> Sapphire', keep it.
    Else, if Gem Type implies Sapphire and Variety is a hue, you can keep the hue or add 'Sapphire'
    based on your project rule. Here we KEEP the suffix to match your requirement.
    """
    gt = _norm_text(gt).title()
    v  = _norm_text(v).title()

    # If Variety already ends with 'Sapphire', keep it as-is.
    if re.search(r"\bSapphire\b$", v):
        return "Sapphire", v

    combined = (gt + " " + v).lower()
    if "sapphire" in combined:
        # If variety includes '<hue> Sapphire' anywhere, normalize spacing and keep it
        m = re.search(r"(padparadscha|teal|pink|yellow|blue|white|green|purple|peach|orange)\s*sapphire", v, flags=re.I)
        if m:
            return "Sapphire", f"{m.group(1).title()} Sapphire"
        # If variety is just a hue word (no suffix), add 'Sapphire'
        m2 = re.search(r"\b(padparadscha|teal|pink|yellow|blue|white|green|purple|peach|orange)\b", v, flags=re.I)
        if m2:
            return "Sapphire", f"{m2.group(1).title()} Sapphire"
        # otherwise: if Gem Type says Sapphire but no clear hue, set Variety='Sapphire'
        return "Sapphire", "Sapphire"

    # Non-sapphire: leave as-is
    return gt or v, v

# Apply the split function to the DataFrame
df["Gem Type"], df["Variety"] = zip(*df.apply(lambda r: split_sapphire(r["Gem Type"], r["Variety"]), axis=1))

# Move treatments out of Variety
def pop_treat_from_variety(variety, treatments):
    # Safely coerce NaNs -> ""
    v_raw = "" if pd.isna(variety) else _norm_text(variety)
    t_raw = "" if pd.isna(treatments) else _norm_text(treatments)

    v = v_raw
    t = t_raw.lower()

    found = None
    if re.search(r"\bunheated|no heat|none\b", v, flags=re.I):
        found = "Unheated"
    elif re.search(r"\bheated|heat\b", v, flags=re.I):
        found = "Heated"
    elif re.search(r"\bberyl|be\s*diffus\b", v, flags=re.I):
        found = "Be Diffusion"
    elif re.search(r"\bdiffus", v, flags=re.I):
        found = "Diffusion"
    elif re.search(r"\boiled|oiling\b", v, flags=re.I):
        found = "Oiled"
    elif re.search(r"\bfilled|fracture filled\b", v, flags=re.I):
        found = "Fracture Filled"

    if not found:
        # Nothing to move; return as cleaned titles
        return v.strip().title(), t_raw.strip().title()

    # Remove treatment words from Variety
    v = re.sub(
        r"\b(unheated|no heat|none|heated|heat|beryllium|be diffusion|diffusion|oiled|oiling|filled|fracture filled)\b",
        "",
        v,
        flags=re.I,
    )
    v = re.sub(r"\s+", " ", v).strip().title()

    # If Treatments empty, fill with the found label
    if t_raw == "":
        t_out = found
    else:
        t_out = t_raw.strip().title()

    return v, t_out

def norm_shape(s):
    """
    Normalize gem cut/shape names to a small, consistent set.
    Uses your _norm_text() to sanitize input first.
    """
    s = _norm_text(s)
    if pd.isna(s) or s == "":
        return np.nan

    low = s.lower()

    # Canonical replacements
    canon_map = {
        "round brilliant": "Round",
        "round": "Round",
        "oval": "Oval",
        "cushion": "Cushion",
        "pear": "Pear",
        "emerald cut": "Emerald Cut",
        "octagon": "Emerald Cut",
        "radiant": "Radiant",
        "princess": "Princess",
        "marquise": "Marquise",
        "asscher": "Asscher",
        "heart": "Heart",
        "trillion": "Trillion",
        "triangle": "Trillion",
        "square": "Square",
        "step cut": "Step Cut",
        "mixed cut": "Mixed Cut",
        "cabochon": "Cabochon"
    }

    for key, val in canon_map.items():
        if key in low:
            return val

    # If no match, return title-cased cleaned string
    return s.title()

# Normalize clarity
def norm_clarity(c):
    """
    Normalize clarity to short grades used in your project:
    {LC, EC, IF, VVS, VS, SI, I} plus optional subgrades {VS1, VS2, SI1, SI2, P1, P2}.
    Returns np.nan if nothing recognizable.
    """
    c = _norm_text(c)
    if pd.isna(c) or c == "": 
        return np.nan
    up = c.upper()

    # Common phrases → codes
    repl = {
        "EYE-CLEAN": "EC", "EYE CLEAN": "EC", "EYE  CLEAN": "EC",
        "LOUPE-CLEAN": "LC", "LOUPE CLEAN": "LC",
        "INTERNALLY FLAWLESS": "IF", "FLAWLESS": "IF", "FL": "IF",
        "VERY VERY SLIGHTLY INCLUDED": "VVS", "VVS1": "VVS", "VVS2": "VVS",
        "VERY SLIGHTLY INCLUDED": "VS", "VS1": "VS1", "VS2": "VS2",
        "SLIGHTLY INCLUDED": "SI", "SI1": "SI1", "SI2": "SI2",
        "INCLUDED": "I", "P1": "P1", "P2": "P2",
        "CLEAN": "EC"
    }
    for k, v in repl.items():
        if k in up:
            # keep explicit subgrades if present
            if v in {"VS1","VS2","SI1","SI2","P1","P2"}:
                return v
            return v

    # Direct code match (e.g., "VVS", "VS", "SI", "I", "EC", "LC", "IF")
    m = re.search(r"\b(LC|EC|IF|VVS|VS|SI|I|VS1|VS2|SI1|SI2|P1|P2)\b", up)
    if m:
        return m.group(1)

    # Transparency words are NOT clarity → treat as unknown
    if re.search(r"\b(TRANSLUCENT|OPAQUE|SEMI[-\s]*TRANSPARENT|TRANSPARENT)\b", up):
        return np.nan

    return np.nan

# Normalize treatments
def norm_treatment(t):
    """
    Canonicalize treatment descriptions to one of:
    {Unheated, Heated, Be Diffusion, Diffusion, Oiled, Fracture Filled}
    Fallback: cleaned, title-cased text if something else (e.g., HPHT, Irradiated, Coated).
    """
    t = _norm_text(t)
    if pd.isna(t) or t == "":
        return np.nan
    s = t.lower()

    # strip obvious non-treatment tails
    s = re.sub(r"(certificate.*|videos?.*|video.*|upon request.*|lab report.*)$", "", s).strip()

    # Order matters: detect "no heat" BEFORE generic "heat"
    if re.search(r"\b(no\s*heat|unheated|none|no treatment|no indications? of heat)\b", s):
        return "Unheated"

    if re.search(r"\b(heated|heat[-\s]?treated|thermal|routine heat|ht)\b", s):
        return "Heated"

    # Diffusion (with/without Be/beryllium)
    if re.search(r"\b(beryl|beryllium|be\s*diffus)\b", s):
        return "Be Diffusion"
    if re.search(r"\b(diffus(ion)?|lattice diffusion|surface diffusion)\b", s):
        return "Diffusion"

    # Oiling
    if re.search(r"\b(oil(ed|ing)?|minor oil|moderate oil|oil present)\b", s):
        return "Oiled"

    # Filling / clarity enhancement
    if re.search(r"\b(filled|fracture\s*filled|glass\s*filled|lead\s*glass|resin\s*filled|fissure\s*filled|clarity\s*enhanced|ce)\b", s):
        return "Fracture Filled"

    # Other less-common treatments (keep explicit text)
    for other in ("hpht", "irradiat", "coat", "surface coat"):
        if other in s:
            return _norm_text(s.title())

    # Fallback to cleaned title-case text (rare descriptions)
    return _norm_text(s.title())

df["Variety"], df["Treatments"] = zip(*df.apply(
    lambda r: pop_treat_from_variety(r["Variety"], r["Treatments"]), axis=1
))
df["Treatments"] = df["Treatments"].map(norm_treatment)

# Clarity
df["Clarity"] = df["Clarity"].map(norm_clarity)

# Shape
df["Cut/Shape"] = df["Cut/Shape"].map(norm_shape)

# Colour
df["Colour"] = df["Colour"].str.title()

# Prices
df["Asking Price (LKR)"] = df["Asking Price (LKR)"].map(_to_float)
df.loc[df["Asking Price (LKR)"]<=0, "Asking Price (LKR)"] = np.nan
df["Final Selling Price (if available)"] = df["Final Selling Price (if available)"].map(_to_float)

# Dimensions 
dims = df["Dimensions (LxWxH, mm)"].map(parse_dimensions_mm)   # reuse your parser
L = dims.map(lambda t: t[0]); W = dims.map(lambda t: t[1]); H = dims.map(lambda t: t[2])
def fmt_dims(l,w,h):
    vals = [v for v in (l,w,h) if pd.notna(v)]
    if not vals: return ""
    return " x ".join([("{:.2f}".format(v)).rstrip("0").rstrip(".") for v in vals]) + " mm"
df["Dimensions (LxWxH, mm)"] = [fmt_dims(l,w,h) for l,w,h in zip(L,W,H)]

# Certification 
df["Certification Status"] = df["Certification Status"].str.lower().map(lambda s: "Yes" if s in ("yes","y","1","true") else "No")

# Date 
df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")

# Carat Weight 
df["Carat Weight (ct)"] = df["Carat Weight (ct)"].map(_to_float).round(2)

# Drop rows with no Gem Type + no Dimensions 
mask_empty = (df["Gem Type"].eq("")) & (df["Dimensions (LxWxH, mm)"].eq(""))
df = df[~mask_empty].reset_index(drop=True)

# Further Colour cleanup
df["Colour"] = df["Colour"].str.replace(r"\bshape\b", "", case=False, regex=True)
df["Colour"] = df["Colour"].str.replace(r"\s+", " ", regex=True).str.strip()

df["Colour"] = df["Colour"].str.replace(r"\bSize\b", "", case=False, regex=True)

# Drop rows with colour 'Multicolor'
df = df[~df["Colour"].str.lower().str.contains(r"multi[- ]?color|mixed", na=False)].copy()

# SAVE 
df.to_csv(PATH_MASTER_CLEAN_DATASET, index=False)
print(f"Saved -> {PATH_MASTER_CLEAN_DATASET}  |  rows: {len(df)}")

Saved -> data/master_gem_schema_clean.csv  |  rows: 596
